# Mana bench: a llama-server on a Colab GPU

Serves **the same model file** the local bench uses (Qwen3-Coder-30B-A3B-Instruct Q4_K_M, SHA-256 checked below) with the same
llama.cpp build (b10507, commit 95c409c13) and the same cache/flash-attention/context flags, so a case that fails there is the
model's failure, not the machine's. Run the cells top to bottom with a **GPU runtime (L4 or A100; a T4's 16 GB is too small)**.
The last cell prints the two commands to run on your PC and then stays running; stop it to end the session and the billing.

In [ ]:
import subprocess, shutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout)
mem = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()[0])
assert mem >= 22000, f"only {mem} MiB of GPU memory: pick an L4 or A100 runtime (Runtime > Change runtime type)"

## 1. Drive cache (optional, saves the 18.5 GB download and the build next time)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/mana-bench"
os.makedirs(CACHE, exist_ok=True)
print(os.listdir(CACHE))

## 2. llama.cpp, the local build's commit

In [ ]:
import os, subprocess
COMMIT = "95c409c13"
cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.split()[0].replace(".", "")
BIN = f"{CACHE}/llama-server-{COMMIT}-sm{cap}"
if not os.path.exists(BIN):
    !git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cd /content/llama.cpp && git checkout {COMMIT}
    !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={cap} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
    !cd /content/llama.cpp && cmake --build build --target llama-server -j
    !cp /content/llama.cpp/build/bin/llama-server {BIN}
    # the shared libraries llama-server loads sit beside it
    !cp /content/llama.cpp/build/bin/*.so* {CACHE}/ 2>/dev/null || true
!chmod +x {BIN}
print(subprocess.run([BIN, "--version"], capture_output=True, text=True, env={**os.environ, "LD_LIBRARY_PATH": CACHE}).stderr[-300:])

## 3. The model, checked against the local file's SHA-256

In [ ]:
import hashlib, os
WANT = "fadc3e5f8d42bf7e894a785b05082e47daee4df26680389817e2093056f088ad"
MODEL = f"{CACHE}/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf"
if not os.path.exists(MODEL):
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF", "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf", local_dir="/content/model")
    !cp {p} {MODEL}
h = hashlib.sha256()
with open(MODEL, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == WANT, f"this is not the bench's model file: {h.hexdigest()}"
print("model file matches the local one:", h.hexdigest()[:16], "...")

## 4. Start the server (the bench's flags, all layers on the GPU) and open a tunnel

In [ ]:
import os, secrets, subprocess, time, urllib.request
KEY = secrets.token_urlsafe(24)
env = {**os.environ, "LD_LIBRARY_PATH": CACHE}
help_text = subprocess.run([BIN, "--help"], capture_output=True, text=True, env=env)
help_text = help_text.stdout + help_text.stderr
args = [BIN, "-m", MODEL, "--host", "127.0.0.1", "--port", "8097", "-c", "32768", "--no-webui", "--cache-ram", "0",
        "-t", str(os.cpu_count() or 4), "-ngl", "99", "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0", "--api-key", KEY]
if "--load-mode" in help_text:
    args += ["--load-mode", "none"]
server = subprocess.Popen(args, stdout=open("/content/llama-server.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(300):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8097/health", timeout=2).status == 200:
            break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("llama-server exited:\n" + open("/content/llama-server.log").read()[-2000:])
    time.sleep(2)
else:
    raise RuntimeError("llama-server didn't come up in 10 minutes")
print("llama-server is up")

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8097", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
import re
URL = None
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0)
        break
print()
print("On your PC, in two terminals, from Mana's node-bot folder:")
print(f"  node bench/remote/llama-proxy.js --remote {URL} --key {KEY} --model D:\\Mana\\tools\\llama\\gguf-models\\Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf")
print("  node bench/self-work-bench.js --adopt --context 32768 --model D:\\Mana\\tools\\llama\\gguf-models\\Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf ...")

## 5. Keep running (stop this cell to end the session; also Runtime > Disconnect and delete runtime)

In [ ]:
import time, urllib.request
t0 = time.time()
while True:
    try:
        ok = urllib.request.urlopen("http://127.0.0.1:8097/health", timeout=5).status == 200
    except Exception as e:
        ok = f"down ({e})"
    print(f"{(time.time() - t0) / 60:6.1f} min  server healthy: {ok}", flush=True)
    time.sleep(120)